In [1]:
import pandas as pd
import numpy as np

# Load both years
df_1 = pd.read_csv('../data/online_retail_09_10.csv', encoding='utf-8')
df_2 = pd.read_csv('../data/online_retail_10_11.csv', encoding='utf-8')

# Combine into one dataframe
df = pd.concat([df_1, df_2], ignore_index=True)

print(df.shape)
df.head()

(1067371, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,12/1/2009 7:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,12/1/2009 7:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,12/1/2009 7:45,1.25,13085.0,United Kingdom


In [2]:
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   InvoiceNo    1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   UnitPrice    1067371 non-null  float64
 6   CustomerID   824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB

Missing values per column:
InvoiceNo           0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     243007
Country             0
dtype: int64

Duplicate rows: 34335


In [3]:
cancellations = df[df['InvoiceNo'].astype(str).str.startswith('C')]
print("Cancellation rows:", len(cancellations))
print(cancellations[['InvoiceNo','Quantity','UnitPrice']].head())

Cancellation rows: 19494
    InvoiceNo  Quantity  UnitPrice
178   C489449       -12       2.95
179   C489449        -6       1.65
180   C489449        -4       4.25
181   C489449        -6       2.10
182   C489449       -12       2.95


In [4]:
print("Negative quantity rows:", (df['Quantity'] < 0).sum())
print("Zero quantity rows:", (df['Quantity'] == 0).sum())

Negative quantity rows: 22950
Zero quantity rows: 0


In [5]:
print("Negative price rows:", (df['UnitPrice'] < 0).sum())
print("Zero price rows:", (df['UnitPrice'] == 0).sum())

Negative price rows: 5
Zero price rows: 6202


In [6]:
print(cancellations['Quantity'].describe())

count    19494.000000
mean       -25.186827
std        805.104908
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max          1.000000
Name: Quantity, dtype: float64


In [7]:
print(df['StockCode'].value_counts().head(20))

StockCode
85123A    5829
22423     4424
85099B    4216
21212     3318
20725     3259
84879     2960
47566     2768
21232     2747
22197     2549
22383     2540
20727     2529
21931     2434
22386     2347
22469     2325
22411     2297
84991     2271
22382     2251
22384     2230
21080     2224
22086     2217
Name: count, dtype: int64


In [8]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())

2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [9]:
non_cancel_negative = df[(df['Quantity'] < 0) & (~df['InvoiceNo'].astype(str).str.startswith('C'))]
print("Count:", len(non_cancel_negative))
print(non_cancel_negative[['InvoiceNo','StockCode','Description','Quantity','UnitPrice']].head(10))

Count: 3457
     InvoiceNo StockCode      Description  Quantity  UnitPrice
263     489464     21733     85123a mixed       -96        0.0
283     489463     71477            short      -240        0.0
284     489467    85123A      21733 mixed      -192        0.0
470     489521     21646              NaN       -50        0.0
3114    489655     20683              NaN       -44        0.0
3162    489660     35956             lost     -1043        0.0
3168    489663    35605A          damages      -117        0.0
4296    489806     18010              NaN      -770        0.0
4538    489820     21133  invcd as 84879?      -720        0.0
4566    489821    85049G              NaN      -240        0.0


In [10]:
odd_cancel = cancellations[cancellations['Quantity'] > 0]
odd_cancel

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
76799,C496350,M,Manual,1,2/1/2010 8:24,373.57,NaN,United Kingdom


In [11]:
df[df['UnitPrice'] < 0][['InvoiceNo','StockCode','Description','Quantity','UnitPrice']]


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
179403,A506401,B,Adjust bad debt,1,-53594.36
276274,A516228,B,Adjust bad debt,1,-44031.79
403472,A528059,B,Adjust bad debt,1,-38925.87
825444,A563186,B,Adjust bad debt,1,-11062.06
825445,A563187,B,Adjust bad debt,1,-11062.06


In [12]:
df[df['UnitPrice'] == 0]['Description'].value_counts().head(15)

Description
check                           162
?                                92
damages                          84
damaged                          81
found                            28
missing                          27
sold as set on dotcom            20
Damaged                          17
adjustment                       16
OWL DOORSTOP                     15
POLYESTER FILLER PAD 45x45cm     12
dotcom                           12
amazon                           11
POLYESTER FILLER PAD 40x40cm     10
IVORY KITCHEN SCALES             10
Name: count, dtype: int64

In [13]:
non_product_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'DOT', 'CRUK', 'C2', 'PADS', 'AMAZONFEE']
for code in non_product_codes:
    count = (df['StockCode'] == code).sum()
    if count > 0:
        print(f"{code}: {count} rows")

POST: 2122 rows
D: 177 rows
M: 1421 rows
BANK CHARGES: 102 rows
DOT: 1446 rows
CRUK: 16 rows
C2: 282 rows
PADS: 19 rows
AMAZONFEE: 43 rows


In [14]:
non_product_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'DOT', 'CRUK', 'C2', 'PADS', 'AMAZONFEE', 'B']
df_clean = df[~df['StockCode'].isin(non_product_codes)].copy()
print("Rows after removing non-product codes:", len(df_clean))

Rows after removing non-product codes: 1061737


In [15]:
df_clean['IsCancellation'] = df_clean['InvoiceNo'].astype(str).str.startswith('C')
print(df_clean['IsCancellation'].value_counts())

IsCancellation
False    1043315
True       18422
Name: count, dtype: int64


In [16]:
df_clean = df_clean[df_clean['UnitPrice'] > 0].copy()
print("Rows after removing zero/negative price:", len(df_clean))

Rows after removing zero/negative price: 1055560


In [17]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Removed {before - len(df_clean)} duplicate rows")

Removed 34038 duplicate rows


In [18]:
df_clean['CustomerID'] = df_clean['CustomerID'].fillna(-1).astype(int)
# -1 acts as a "Guest/Unknown" customer flag for revenue-level analysis,
# while customer-level analysis (RFM, cohorts) will explicitly EXCLUDE CustomerID == -1
print(df_clean['CustomerID'].value_counts().head())

CustomerID
-1        227279
 17841     12623
 14911     11255
 12748      6626
 14606      6486
Name: count, dtype: int64


In [19]:
print("Final shape:", df_clean.shape)
print("\nRemaining nulls:\n", df_clean.isnull().sum())

Final shape: (1021522, 9)

Remaining nulls:
 InvoiceNo         0
StockCode         0
Description       0
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Country           0
IsCancellation    0
dtype: int64


In [20]:
dim_products = df_clean[['StockCode', 'Description']].drop_duplicates(subset='StockCode', keep='first').reset_index(drop=True)
dim_products['ProductKey'] = dim_products.index + 1  # surrogate key
print(dim_products.shape)
dim_products.head()

(4922, 3)


,StockCode,Description,ProductKey
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,1
1,79323P,PINK CHERRY LIGHTS,2
2,79323W,WHITE CHERRY LIGHTS,3
3,22041,"RECORD FRAME 7"" SINGLE SIZE",4
4,21232,STRAWBERRY CERAMIC TRINKET BOX,5


In [21]:
dim_customers = df_clean[['CustomerID', 'Country']].drop_duplicates(subset='CustomerID', keep='first').reset_index(drop=True)
dim_customers['CustomerKey'] = dim_customers.index + 1
print(dim_customers.shape)
dim_customers.head()

(5895, 3)


,CustomerID,Country,CustomerKey
0,13085,United Kingdom,1
1,13078,United Kingdom,2
2,15362,United Kingdom,3
3,18102,United Kingdom,4
4,12682,France,5


In [22]:
date_range = pd.date_range(start=df_clean['InvoiceDate'].min().date(), end=df_clean['InvoiceDate'].max().date(), freq='D')
dim_dates = pd.DataFrame({'FullDate': date_range})
dim_dates['DateKey'] = dim_dates['FullDate'].dt.strftime('%Y%m%d').astype(int)
dim_dates['Year'] = dim_dates['FullDate'].dt.year
dim_dates['Month'] = dim_dates['FullDate'].dt.month
dim_dates['MonthName'] = dim_dates['FullDate'].dt.month_name()
dim_dates['Quarter'] = dim_dates['FullDate'].dt.quarter
dim_dates['DayOfWeek'] = dim_dates['FullDate'].dt.day_name()
dim_dates['IsWeekend'] = dim_dates['FullDate'].dt.dayofweek >= 5
print(dim_dates.shape)
dim_dates.head()

(739, 8)


,FullDate,DateKey,Year,Month,MonthName,Quarter,DayOfWeek,IsWeekend
0,2009-12-01,20091201,2009,12,December,4,Tuesday,False
1,2009-12-02,20091202,2009,12,December,4,Wednesday,False
2,2009-12-03,20091203,2009,12,December,4,Thursday,False
3,2009-12-04,20091204,2009,12,December,4,Friday,False
4,2009-12-05,20091205,2009,12,December,4,Saturday,True


In [23]:
fact_sales = df_clean.merge(dim_products[['StockCode', 'ProductKey']], on='StockCode', how='left')
fact_sales = fact_sales.merge(dim_customers[['CustomerID', 'CustomerKey']], on='CustomerID', how='left')
fact_sales['DateKey'] = fact_sales['InvoiceDate'].dt.strftime('%Y%m%d').astype(int)

fact_sales['TotalPrice'] = fact_sales['Quantity'] * fact_sales['UnitPrice']  # revenue measure

fact_sales = fact_sales[[
    'InvoiceNo', 'ProductKey', 'CustomerKey', 'DateKey',
    'Quantity', 'UnitPrice', 'TotalPrice', 'IsCancellation'
]]
print(fact_sales.shape)
fact_sales.head()

(1021522, 8)


,InvoiceNo,ProductKey,CustomerKey,DateKey,Quantity,UnitPrice,TotalPrice,IsCancellation
0,489434,1,1,20091201,12,6.95,83.4,False
1,489434,2,1,20091201,12,6.75,81.0,False
2,489434,3,1,20091201,12,6.75,81.0,False
3,489434,4,1,20091201,48,2.10,100.8,False
4,489434,5,1,20091201,24,1.25,30.0,False


In [24]:
print("fact_sales rows:", len(fact_sales))
print("dim_products rows:", len(dim_products))
print("dim_customers rows:", len(dim_customers))
print("dim_dates rows:", len(dim_dates))
print("\nAny nulls in fact_sales?")
print(fact_sales.isnull().sum())
print("\nTotal revenue (sanity check):", fact_sales['TotalPrice'].sum())

fact_sales rows: 1021522
dim_products rows: 4922
dim_customers rows: 5895
dim_dates rows: 739

Any nulls in fact_sales?
InvoiceNo         0
ProductKey        0
CustomerKey       0
DateKey           0
Quantity          0
UnitPrice         0
TotalPrice        0
IsCancellation    0
dtype: int64

Total revenue (sanity check): 18930915.990000002


In [25]:
import sqlite3

conn = sqlite3.connect('../sql/retail_analytics.db')

In [26]:
dim_products.to_sql('dim_products', conn, if_exists='replace', index=False)
dim_customers.to_sql('dim_customers', conn, if_exists='replace', index=False)
dim_dates.to_sql('dim_dates', conn, if_exists='replace', index=False)
fact_sales.to_sql('fact_sales', conn, if_exists='replace', index=False)

print("Tables loaded successfully")

Tables loaded successfully


In [27]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

for t in ['dim_products', 'dim_customers', 'dim_dates', 'fact_sales']:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {t}", conn)
    print(t, count['cnt'].values[0])

            name
0   dim_products
1  dim_customers
2      dim_dates
3     fact_sales
dim_products 4922
dim_customers 5895
dim_dates 739
fact_sales 1021522


In [28]:
cur = conn.cursor()
cur.execute("CREATE INDEX IF NOT EXISTS idx_fact_product ON fact_sales(ProductKey);")
cur.execute("CREATE INDEX IF NOT EXISTS idx_fact_customer ON fact_sales(CustomerKey);")
cur.execute("CREATE INDEX IF NOT EXISTS idx_fact_date ON fact_sales(DateKey);")
conn.commit()
print("Indexes created")

Indexes created


In [29]:
test_query = """
SELECT 
    d.Year, 
    d.MonthName, 
    ROUND(SUM(f.TotalPrice), 2) AS Revenue
FROM fact_sales f
JOIN dim_dates d ON f.DateKey = d.DateKey
WHERE f.IsCancellation = 0
GROUP BY d.Year, d.MonthName
ORDER BY d.Year, MIN(d.FullDate)
LIMIT 12;
"""
result = pd.read_sql(test_query, conn)
print(result)

    Year  MonthName     Revenue
0   2009   December   798232.03
1   2010    January   621353.43
2   2010   February   537932.64
3   2010      March   761748.53
4   2010      April   646551.86
5   2010        May   643585.64
6   2010       June   697384.87
7   2010       July   633079.42
8   2010     August   674192.89
9   2010  September   869277.16
10  2010    October  1094563.63
11  2010   November  1429751.84


The business question: "Is revenue growing or shrinking month to month, and by how much?"

In [30]:
query1 = """
WITH monthly_revenue AS (
    SELECT 
        d.Year,
        d.Month,
        d.MonthName,
        ROUND(SUM(f.TotalPrice), 2) AS Revenue
    FROM fact_sales f
    JOIN dim_dates d ON f.DateKey = d.DateKey
    WHERE f.IsCancellation = 0
    GROUP BY d.Year, d.Month
)
SELECT 
    Year,
    MonthName,
    Revenue,
    LAG(Revenue) OVER (ORDER BY Year, Month) AS PrevMonthRevenue,
    ROUND(
        (Revenue - LAG(Revenue) OVER (ORDER BY Year, Month)) 
        / LAG(Revenue) OVER (ORDER BY Year, Month) * 100, 2
    ) AS MoM_Growth_Pct
FROM monthly_revenue
ORDER BY Year, Month;
"""
result1 = pd.read_sql(query1, conn)
print(result1)

    Year  MonthName     Revenue  PrevMonthRevenue  MoM_Growth_Pct
0   2009   December   798232.03               NaN             NaN
1   2010    January   621353.43         798232.03          -22.16
2   2010   February   537932.64         621353.43          -13.43
3   2010      March   761748.53         537932.64           41.61
4   2010      April   646551.86         761748.53          -15.12
5   2010        May   643585.64         646551.86           -0.46
6   2010       June   697384.87         643585.64            8.36
7   2010       July   633079.42         697384.87           -9.22
8   2010     August   674192.89         633079.42            6.49
9   2010  September   869277.16         674192.89           28.94
10  2010    October  1094563.63         869277.16           25.92
11  2010   November  1429751.84        1094563.63           30.62
12  2010   December   775717.50        1429751.84          -45.74
13  2011    January   670439.46         775717.50          -13.57
14  2011  

The business question: "What's our cumulative revenue so far this fiscal period — are we on track for the year?"

In [31]:
query2 = """
WITH monthly_revenue AS (
    SELECT 
        d.Year,
        d.Month,
        d.MonthName,
        ROUND(SUM(f.TotalPrice), 2) AS Revenue
    FROM fact_sales f
    JOIN dim_dates d ON f.DateKey = d.DateKey
    WHERE f.IsCancellation = 0
    GROUP BY d.Year, d.Month
)
SELECT 
    Year,
    MonthName,
    Revenue,
    ROUND(SUM(Revenue) OVER (ORDER BY Year, Month), 2) AS RunningTotalRevenue
FROM monthly_revenue
ORDER BY Year, Month;
"""
result2 = pd.read_sql(query2, conn)
print(result2)

    Year  MonthName     Revenue  RunningTotalRevenue
0   2009   December   798232.03            798232.03
1   2010    January   621353.43           1419585.46
2   2010   February   537932.64           1957518.10
3   2010      March   761748.53           2719266.63
4   2010      April   646551.86           3365818.49
5   2010        May   643585.64           4009404.13
6   2010       June   697384.87           4706789.00
7   2010       July   633079.42           5339868.42
8   2010     August   674192.89           6014061.31
9   2010  September   869277.16           6883338.47
10  2010    October  1094563.63           7977902.10
11  2010   November  1429751.84           9407653.94
12  2010   December   775717.50          10183371.44
13  2011    January   670439.46          10853810.90
14  2011   February   507866.54          11361677.44
15  2011      March   689841.84          12051519.28
16  2011      April   515499.66          12567018.94
17  2011        May   740036.33          13307

In [32]:
query2b = """
WITH monthly_revenue AS (
    SELECT 
        d.Year,
        d.Month,
        d.MonthName,
        ROUND(SUM(f.TotalPrice), 2) AS Revenue
    FROM fact_sales f
    JOIN dim_dates d ON f.DateKey = d.DateKey
    WHERE f.IsCancellation = 0
    GROUP BY d.Year, d.Month
)
SELECT 
    Year,
    MonthName,
    Revenue,
    ROUND(SUM(Revenue) OVER (PARTITION BY Year ORDER BY Month), 2) AS YTD_Revenue
FROM monthly_revenue
ORDER BY Year, Month;
"""
result2b = pd.read_sql(query2b, conn)
print(result2b)

    Year  MonthName     Revenue  YTD_Revenue
0   2009   December   798232.03    798232.03
1   2010    January   621353.43    621353.43
2   2010   February   537932.64   1159286.07
3   2010      March   761748.53   1921034.60
4   2010      April   646551.86   2567586.46
5   2010        May   643585.64   3211172.10
6   2010       June   697384.87   3908556.97
7   2010       July   633079.42   4541636.39
8   2010     August   674192.89   5215829.28
9   2010  September   869277.16   6085106.44
10  2010    October  1094563.63   7179670.07
11  2010   November  1429751.84   8609421.91
12  2010   December   775717.50   9385139.41
13  2011    January   670439.46    670439.46
14  2011   February   507866.54   1178306.00
15  2011      March   689841.84   1868147.84
16  2011      April   515499.66   2383647.50
17  2011        May   740036.33   3123683.83
18  2011       June   737683.99   3861367.82
19  2011       July   688252.67   4549620.49
20  2011     August   724308.16   5273928.65
21  2011  

The business question: "Who are our top 20 customers by total revenue, and how are they ranked relative to each other?"

In [33]:
query3 = """
WITH customer_revenue AS (
    SELECT 
        f.CustomerKey,
        c.CustomerID,
        c.Country,
        ROUND(SUM(f.TotalPrice), 2) AS TotalSpend
    FROM fact_sales f
    JOIN dim_customers c ON f.CustomerKey = c.CustomerKey
    WHERE f.IsCancellation = 0
      AND c.CustomerID != -1   -- exclude guest checkouts, we can't track them individually
    GROUP BY f.CustomerKey
)
SELECT 
    CustomerID,
    Country,
    TotalSpend,
    ROW_NUMBER() OVER (ORDER BY TotalSpend DESC) AS RowNum,
    RANK() OVER (ORDER BY TotalSpend DESC) AS Rank_,
    DENSE_RANK() OVER (ORDER BY TotalSpend DESC) AS DenseRank
FROM customer_revenue
ORDER BY TotalSpend DESC
LIMIT 20;
"""
result3 = pd.read_sql(query3, conn)
print(result3)

    CustomerID         Country  TotalSpend  RowNum  Rank_  DenseRank
0        18102  United Kingdom   580987.04       1      1          1
1        14646     Netherlands   526751.52       2      2          2
2        14156            EIRE   303609.17       3      3          3
3        14911            EIRE   272801.15       4      4          4
4        17450  United Kingdom   244784.25       5      5          5
5        13694  United Kingdom   195640.69       6      6          6
6        17511  United Kingdom   172132.87       7      7          7
7        16446  United Kingdom   168472.50       8      8          8
8        16684  United Kingdom   147142.77       9      9          9
9        12415       Australia   144033.37      10     10         10
10       15061  United Kingdom   126387.57      11     11         11
11       15311  United Kingdom   114671.42      12     12         12
12       13089  United Kingdom   113416.91      13     13         13
13       16029  United Kingdom   1

The business question: "How much does a typical order cost, and how much is a customer worth over their lifetime with us?"

In [34]:
query4 = """
WITH customer_orders AS (
    SELECT 
        c.CustomerID,
        c.Country,
        COUNT(DISTINCT f.InvoiceNo) AS NumOrders,
        SUM(f.TotalPrice) AS TotalSpend
    FROM fact_sales f
    JOIN dim_customers c ON f.CustomerKey = c.CustomerKey
    WHERE f.IsCancellation = 0
      AND c.CustomerID != -1
    GROUP BY c.CustomerID, c.Country
)
SELECT 
    CustomerID,
    Country,
    NumOrders,
    ROUND(TotalSpend, 2) AS CLV,
    ROUND(TotalSpend / NumOrders, 2) AS AvgOrderValue
FROM customer_orders
ORDER BY CLV DESC
LIMIT 20;
"""
result4 = pd.read_sql(query4, conn)
print(result4)

    CustomerID         Country  NumOrders        CLV  AvgOrderValue
0        18102  United Kingdom        145  580987.04        4006.81
1        14646     Netherlands        145  526751.52        3632.77
2        14156            EIRE        150  303609.17        2024.06
3        14911            EIRE        376  272801.15         725.53
4        17450  United Kingdom         51  244784.25        4799.69
5        13694  United Kingdom        143  195640.69        1368.12
6        17511  United Kingdom         60  172132.87        2868.88
7        16446  United Kingdom          2  168472.50       84236.25
8        16684  United Kingdom         55  147142.77        2675.32
9        12415       Australia         24  144033.37        6001.39
10       15061  United Kingdom        127  126387.57         995.18
11       15311  United Kingdom        207  114671.42         553.97
12       13089  United Kingdom        203  113416.91         558.70
13       16029  United Kingdom        106  10962

In [35]:
query4b = """
SELECT 
    COUNT(DISTINCT f.InvoiceNo) AS TotalOrders,
    ROUND(SUM(f.TotalPrice), 2) AS TotalRevenue,
    ROUND(SUM(f.TotalPrice) / COUNT(DISTINCT f.InvoiceNo), 2) AS StoreWideAOV
FROM fact_sales f
WHERE f.IsCancellation = 0;
"""
result4b = pd.read_sql(query4b, conn)
print(result4b)

   TotalOrders  TotalRevenue  StoreWideAOV
0        39568   19655624.67        496.76


The business question: "If a customer makes their first purchase in a given month, what % of them come back to buy again in each following month?" This is the classic cohort retention curve you see in every SaaS/e-commerce analytics dashboard.

In [36]:
query5a = """
SELECT 
    c.CustomerID,
    MIN(d.FullDate) AS FirstPurchaseDate,
    strftime('%Y-%m', MIN(d.FullDate)) AS CohortMonth
FROM fact_sales f
JOIN dim_customers c ON f.CustomerKey = c.CustomerKey
JOIN dim_dates d ON f.DateKey = d.DateKey
WHERE f.IsCancellation = 0
  AND c.CustomerID != -1
GROUP BY c.CustomerID
ORDER BY FirstPurchaseDate
LIMIT 10;
"""
result5a = pd.read_sql(query5a, conn)
print(result5a)

   CustomerID    FirstPurchaseDate CohortMonth
0       12490  2009-12-01 00:00:00     2009-12
1       12533  2009-12-01 00:00:00     2009-12
2       12682  2009-12-01 00:00:00     2009-12
3       12758  2009-12-01 00:00:00     2009-12
4       12836  2009-12-01 00:00:00     2009-12
5       12913  2009-12-01 00:00:00     2009-12
6       12921  2009-12-01 00:00:00     2009-12
7       12931  2009-12-01 00:00:00     2009-12
8       13078  2009-12-01 00:00:00     2009-12
9       13085  2009-12-01 00:00:00     2009-12


In [37]:
cur = conn.cursor()
cur.execute("CREATE INDEX IF NOT EXISTS idx_dim_customers_id ON dim_customers(CustomerID);")
conn.commit()

In [39]:
query5b = """
WITH customer_months AS (
    -- one row per customer per month they purchased (not per line item)
    SELECT DISTINCT
        f.CustomerKey,
        CAST(strftime('%Y', d.FullDate) AS INT) AS OrderYear,
        CAST(strftime('%m', d.FullDate) AS INT) AS OrderMonth
    FROM fact_sales f
    JOIN dim_dates d ON f.DateKey = d.DateKey
    WHERE f.IsCancellation = 0
      AND f.CustomerKey != (SELECT CustomerKey FROM dim_customers WHERE CustomerID = -1)
),
first_purchase AS (
    SELECT 
        CustomerKey,
        MIN(OrderYear * 12 + OrderMonth) AS CohortYearMonth
    FROM customer_months
    GROUP BY CustomerKey
)
SELECT 
    cm.CustomerKey,
    cm.OrderYear,
    cm.OrderMonth,
    fp.CohortYearMonth,
    (cm.OrderYear * 12 + cm.OrderMonth) - fp.CohortYearMonth AS MonthsSinceFirstPurchase
FROM customer_months cm
JOIN first_purchase fp ON cm.CustomerKey = fp.CustomerKey
ORDER BY cm.CustomerKey, cm.OrderYear, cm.OrderMonth
LIMIT 15;
"""
result5b = pd.read_sql(query5b, conn)
result5b

,CustomerKey,OrderYear,OrderMonth,CohortYearMonth,MonthsSinceFirstPurchase
0,1,2009,12,24120,0
1,1,2010,1,24120,1
2,1,2011,2,24120,14
3,1,2011,7,24120,19
4,2,2009,12,24120,0
5,2,2010,1,24120,1
6,2,2010,2,24120,2
7,2,2010,3,24120,3
8,2,2010,4,24120,4
9,2,2010,5,24120,5


In [40]:
query5c = """
WITH customer_months AS (
    SELECT DISTINCT
        f.CustomerKey,
        CAST(strftime('%Y', d.FullDate) AS INT) AS OrderYear,
        CAST(strftime('%m', d.FullDate) AS INT) AS OrderMonth
    FROM fact_sales f
    JOIN dim_dates d ON f.DateKey = d.DateKey
    WHERE f.IsCancellation = 0
      AND f.CustomerKey != (SELECT CustomerKey FROM dim_customers WHERE CustomerID = -1)
),
first_purchase AS (
    SELECT 
        CustomerKey,
        MIN(OrderYear * 12 + OrderMonth) AS CohortYearMonth
    FROM customer_months
    GROUP BY CustomerKey
),
cohort_activity AS (
    SELECT 
        cm.CustomerKey,
        fp.CohortYearMonth,
        (cm.OrderYear * 12 + cm.OrderMonth) - fp.CohortYearMonth AS MonthsSinceFirstPurchase
    FROM customer_months cm
    JOIN first_purchase fp ON cm.CustomerKey = fp.CustomerKey
)
SELECT 
    CohortYearMonth,
    MonthsSinceFirstPurchase,
    COUNT(DISTINCT CustomerKey) AS ActiveCustomers
FROM cohort_activity
GROUP BY CohortYearMonth, MonthsSinceFirstPurchase
ORDER BY CohortYearMonth, MonthsSinceFirstPurchase;
"""
result5c = pd.read_sql(query5c, conn)
print(result5c.head(20))
print("\nTotal rows:", len(result5c))

    CohortYearMonth  MonthsSinceFirstPurchase  ActiveCustomers
0             24120                         0              952
1             24120                         1              336
2             24120                         2              317
3             24120                         3              405
4             24120                         4              360
5             24120                         5              342
6             24120                         6              359
7             24120                         7              327
8             24120                         8              321
9             24120                         9              344
10            24120                        10              401
11            24120                        11              472
12            24120                        12              358
13            24120                        13              269
14            24120                        14          

In [41]:
query5d = """
WITH customer_months AS (
    SELECT DISTINCT
        f.CustomerKey,
        CAST(strftime('%Y', d.FullDate) AS INT) AS OrderYear,
        CAST(strftime('%m', d.FullDate) AS INT) AS OrderMonth
    FROM fact_sales f
    JOIN dim_dates d ON f.DateKey = d.DateKey
    WHERE f.IsCancellation = 0
      AND f.CustomerKey != (SELECT CustomerKey FROM dim_customers WHERE CustomerID = -1)
),
first_purchase AS (
    SELECT 
        CustomerKey,
        MIN(OrderYear * 12 + OrderMonth) AS CohortYearMonth
    FROM customer_months
    GROUP BY CustomerKey
),
cohort_activity AS (
    SELECT 
        cm.CustomerKey,
        fp.CohortYearMonth,
        (cm.OrderYear * 12 + cm.OrderMonth) - fp.CohortYearMonth AS MonthsSinceFirstPurchase
    FROM customer_months cm
    JOIN first_purchase fp ON cm.CustomerKey = fp.CustomerKey
),
cohort_counts AS (
    SELECT 
        CohortYearMonth,
        MonthsSinceFirstPurchase,
        COUNT(DISTINCT CustomerKey) AS ActiveCustomers
    FROM cohort_activity
    GROUP BY CohortYearMonth, MonthsSinceFirstPurchase
),
cohort_sizes AS (
    SELECT 
        CohortYearMonth,
        ActiveCustomers AS CohortSize
    FROM cohort_counts
    WHERE MonthsSinceFirstPurchase = 0
)
SELECT 
    cc.CohortYearMonth,
    -- convert back to a readable YYYY-MM label
    (cc.CohortYearMonth - 1) / 12 AS CohortYear,
    ((cc.CohortYearMonth - 1) % 12) + 1 AS CohortMonthNum,
    cc.MonthsSinceFirstPurchase,
    cc.ActiveCustomers,
    cs.CohortSize,
    ROUND(100.0 * cc.ActiveCustomers / cs.CohortSize, 1) AS RetentionPct
FROM cohort_counts cc
JOIN cohort_sizes cs ON cc.CohortYearMonth = cs.CohortYearMonth
ORDER BY cc.CohortYearMonth, cc.MonthsSinceFirstPurchase;
"""
cohort_retention = pd.read_sql(query5d, conn)
print(cohort_retention.head(15))
print("\nTotal rows:", len(cohort_retention))

    CohortYearMonth  CohortYear  CohortMonthNum  MonthsSinceFirstPurchase  \
0             24120        2009              12                         0   
1             24120        2009              12                         1   
2             24120        2009              12                         2   
3             24120        2009              12                         3   
4             24120        2009              12                         4   
5             24120        2009              12                         5   
6             24120        2009              12                         6   
7             24120        2009              12                         7   
8             24120        2009              12                         8   
9             24120        2009              12                         9   
10            24120        2009              12                        10   
11            24120        2009              12                        11   

In [43]:
pivot = cohort_retention.pivot(index=['CohortYear','CohortMonthNum'], columns='MonthsSinceFirstPurchase', values='RetentionPct')
pivot = pivot.iloc[:, :13]  # first 13 months (0-12) for readability
pivot

MonthsSinceFirstPurchase      0     1     2     3     4     5     6     7   \
CohortYear CohortMonthNum                                                    
2009       12              100.0  35.3  33.3  42.5  37.8  35.9  37.7  34.3   
2010       1               100.0  20.7  31.2  30.6  26.4  30.1  25.9  23.0   
           2               100.0  23.5  22.7  29.3  24.5  19.7  19.2  28.8   
           3               100.0  19.1  23.2  24.1  23.2  20.5  24.6  30.5   
           4               100.0  19.1  19.1  16.0  18.4  22.2  27.6  26.3   
           5               100.0  15.7  16.9  17.3  17.7  25.6  21.3  12.6   
           6               100.0  17.6  18.7  20.6  23.2  28.5  12.7   9.0   
           7               100.0  15.7  18.4  29.7  29.2  14.1  11.4  14.6   
           8               100.0  19.8  29.0  32.7  16.7  11.7   9.9  13.0   
           9               100.0  23.0  23.4  13.0   8.8  10.5  13.8   9.6   
           10              100.0  25.9  14.7  12.5   8.8   8.3  13.1  13.9   
           11              100.0  17.5   9.5   9.5   7.7   8.9  12.9  10.1   
           12              100.0   9.2   5.3   9.2  11.8   6.6   5.3   9.2   
2011       1               100.0  16.7  22.2  19.4  22.2  15.3  15.3  12.5   
           2               100.0  16.0  16.0  18.4  21.6  16.0  16.0  15.2   
           3               100.0  17.9  21.8  20.1  22.3  15.1  20.7  20.1   
           4               100.0  24.5  19.8  19.8  17.9  23.6  17.9  25.5   
           5               100.0  23.4  24.3  16.2  21.6  20.7  26.1  11.7   
           6               100.0  23.1  21.3  26.9  20.4  27.8   8.3   NaN   
           7               100.0  20.8  29.7  27.7  34.7  15.8   NaN   NaN   
           8               100.0  26.4  30.2  26.4  17.0   NaN   NaN   NaN   
           9               100.0  27.1  37.8  14.9   NaN   NaN   NaN   NaN   
           10              100.0  31.7  15.8   NaN   NaN   NaN   NaN   NaN   
           11              100.0  14.1   NaN   NaN   NaN   NaN   NaN   NaN   
           12              100.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   

MonthsSinceFirstPurchase     8     9     10    11    12  
CohortYear CohortMonthNum                                
2009       12              33.7  36.1  42.1  49.6  37.6  
2010       1               27.7  31.7  30.1  17.3  22.0  
           2               25.6  27.7  11.5  12.5  15.2  
           3               27.8  10.9  11.4  14.4  20.0  
           4               10.6  10.9   7.5  14.0  14.0  
           5                5.9   8.3  11.4  13.4  15.4  
           6                8.2  11.2  10.5  13.9  15.0  
           7               14.6  11.4  13.5  14.6  13.0  
           8               13.6  13.0  13.0  11.7  14.8  
           9               13.0  13.0  12.1  10.0  22.6  
           10              10.7   9.3  10.7  12.8  19.2  
           11               8.6   9.2  11.0  14.7  25.2  
           12               5.3   2.6   9.2  19.7   2.6  
2011       1               11.1  20.8  25.0  11.1   NaN  
           2               15.2  18.4   6.4   NaN   NaN  
           3               24.0   9.5   NaN   NaN   NaN  
           4               11.3   NaN   NaN   NaN   NaN  
           5                NaN   NaN   NaN   NaN   NaN  
           6                NaN   NaN   NaN   NaN   NaN  
           7                NaN   NaN   NaN   NaN   NaN  
           8                NaN   NaN   NaN   NaN   NaN  
           9                NaN   NaN   NaN   NaN   NaN  
           10               NaN   NaN   NaN   NaN   NaN  
           11               NaN   NaN   NaN   NaN   NaN  
           12               NaN   NaN   NaN   NaN   NaN

In [44]:
cohort_retention.to_sql('cohort_retention', conn, if_exists='replace', index=False)
cohort_retention.to_csv('../data/cohort_retention.csv', index=False)
print("Saved cohort_retention table and CSV")

Saved cohort_retention table and CSV


In [45]:
query6 = """
WITH customer_orders AS (
    SELECT 
        c.CustomerID,
        c.Country,
        MAX(d.FullDate) AS LastPurchaseDate,
        COUNT(DISTINCT f.InvoiceNo) AS Frequency,
        SUM(f.TotalPrice) AS Monetary
    FROM fact_sales f
    JOIN dim_customers c ON f.CustomerKey = c.CustomerKey
    JOIN dim_dates d ON f.DateKey = d.DateKey
    WHERE f.IsCancellation = 0
      AND c.CustomerID != -1
    GROUP BY c.CustomerID, c.Country
),
rfm_base AS (
    SELECT 
        CustomerID,
        Country,
        CAST(julianday('2011-12-10') - julianday(LastPurchaseDate) AS INT) AS Recency,
        Frequency,
        ROUND(Monetary, 2) AS Monetary
    FROM customer_orders
),
rfm_scored AS (
    SELECT 
        CustomerID,
        Country,
        Recency,
        Frequency,
        Monetary,
        -- Recency: LOWER days = BETTER, so reverse the scoring (most recent = 5)
        6 - NTILE(5) OVER (ORDER BY Recency) AS R_Score,
        NTILE(5) OVER (ORDER BY Frequency) AS F_Score,
        NTILE(5) OVER (ORDER BY Monetary) AS M_Score
    FROM rfm_base
)
SELECT 
    *,
    (R_Score + F_Score + M_Score) AS RFM_Total,
    CAST(R_Score AS TEXT) || CAST(F_Score AS TEXT) || CAST(M_Score AS TEXT) AS RFM_Code
FROM rfm_scored
ORDER BY RFM_Total DESC
LIMIT 20;
"""
rfm_result = pd.read_sql(query6, conn)
print(rfm_result)

    CustomerID         Country  Recency  Frequency  Monetary  R_Score  \
0        12433          Norway        1         10  16794.14        5   
1        14441  United Kingdom        1         11   4611.69        5   
2        15804  United Kingdom        1         13   4206.39        5   
3        17754  United Kingdom        1         13   4451.22        5   
4        16626  United Kingdom        1         17   4413.10        5   
5        17001  United Kingdom        1         19   6640.80        5   
6        14422  United Kingdom        1         19  10271.52        5   
7        16954  United Kingdom        1         20   6081.88        5   
8        12662         Germany        1         20   6130.54        5   
9        13426  United Kingdom        1         22   8167.38        5   
10       15694  United Kingdom        1         23   9963.48        5   
11       17364  United Kingdom        1         28  11972.92        5   
12       13113  United Kingdom        1         37 

In [46]:
query7 = """
WITH customer_orders AS (
    SELECT 
        c.CustomerID,
        c.Country,
        MAX(d.FullDate) AS LastPurchaseDate,
        COUNT(DISTINCT f.InvoiceNo) AS Frequency,
        SUM(f.TotalPrice) AS Monetary
    FROM fact_sales f
    JOIN dim_customers c ON f.CustomerKey = c.CustomerKey
    JOIN dim_dates d ON f.DateKey = d.DateKey
    WHERE f.IsCancellation = 0
      AND c.CustomerID != -1
    GROUP BY c.CustomerID, c.Country
),
rfm_base AS (
    SELECT 
        CustomerID,
        Country,
        CAST(julianday('2011-12-10') - julianday(LastPurchaseDate) AS INT) AS Recency,
        Frequency,
        ROUND(Monetary, 2) AS Monetary
    FROM customer_orders
),
rfm_scored AS (
    SELECT 
        CustomerID,
        Country,
        Recency,
        Frequency,
        Monetary,
        6 - NTILE(5) OVER (ORDER BY Recency) AS R_Score,
        NTILE(5) OVER (ORDER BY Frequency) AS F_Score,
        NTILE(5) OVER (ORDER BY Monetary) AS M_Score
    FROM rfm_base
),
rfm_segmented AS (
    SELECT 
        *,
        CASE
            WHEN R_Score >= 4 AND F_Score >= 4 AND M_Score >= 4 THEN 'Champions'
            WHEN R_Score >= 3 AND F_Score >= 4 THEN 'Loyal Customers'
            WHEN M_Score >= 4 AND F_Score < 4 THEN 'Big Spenders'
            WHEN R_Score <= 2 AND F_Score >= 3 THEN 'At Risk'
            WHEN R_Score <= 2 AND F_Score <= 2 AND M_Score <= 2 THEN 'Lost'
            ELSE 'New / Others'
        END AS Segment
    FROM rfm_scored
)
SELECT * FROM rfm_segmented;
"""
rfm_full = pd.read_sql(query7, conn)
print(rfm_full['Segment'].value_counts())
print("\nTotal customers:", len(rfm_full))

Segment
New / Others       1497
Lost               1358
Champions          1333
At Risk             714
Loyal Customers     660
Big Spenders        299
Name: count, dtype: int64

Total customers: 5861


In [47]:
segment_revenue = rfm_full.groupby('Segment').agg(
    NumCustomers=('CustomerID', 'count'),
    TotalRevenue=('Monetary', 'sum')
).reset_index()

segment_revenue['PctOfCustomers'] = round(100 * segment_revenue['NumCustomers'] / segment_revenue['NumCustomers'].sum(), 1)
segment_revenue['PctOfRevenue'] = round(100 * segment_revenue['TotalRevenue'] / segment_revenue['TotalRevenue'].sum(), 1)

segment_revenue = segment_revenue.sort_values('PctOfRevenue', ascending=False)
print(segment_revenue)

           Segment  NumCustomers  TotalRevenue  PctOfCustomers  PctOfRevenue
2        Champions          1333   11719112.76            22.7          68.6
4  Loyal Customers           660    1878452.01            11.3          11.0
0          At Risk           714    1423444.41            12.2           8.3
1     Big Spenders           299     890086.70             5.1           5.2
5     New / Others          1497     814004.02            25.5           4.8
3             Lost          1358     347978.39            23.2           2.0


In [48]:
import os

export_folder = '../data/powerbi_export'
os.makedirs(export_folder, exist_ok=True)

tables_to_export = {
    'fact_sales': fact_sales,
    'dim_products': dim_products,
    'dim_customers': dim_customers,
    'dim_dates': dim_dates,
    'cohort_retention': cohort_retention,
    'rfm_customers': rfm_full,
    'rfm_segment_summary': segment_revenue
}

for name, df_table in tables_to_export.items():
    path = f'{export_folder}/{name}.csv'
    df_table.to_csv(path, index=False)
    print(f"Saved {name}.csv — {df_table.shape[0]} rows")

Saved fact_sales.csv — 1021522 rows
Saved dim_products.csv — 4922 rows
Saved dim_customers.csv — 5895 rows
Saved dim_dates.csv — 739 rows
Saved cohort_retention.csv — 325 rows
Saved rfm_customers.csv — 5861 rows
Saved rfm_segment_summary.csv — 6 rows
